<a href="https://colab.research.google.com/github/paslariirina01-lab/git_workhard/blob/main/%22fine_tuning_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [1]:
#установка библиотеки
!pip install -U transformers

import numpy as np
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)


In [2]:
#загрузка датасета
dataset = load_dataset("ag_news")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [3]:
#токенизация
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [4]:
#инициализация модели и метрика
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
#TrainingArguments и Trainer
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [8]:
#обучение
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.196080,0.173746,0.942105
2,0.130687,0.188658,0.947368
3,0.086505,0.220376,0.946053


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=22500, training_loss=0.14267427842881944, metrics={'train_runtime': 4582.6282, 'train_samples_per_second': 78.558, 'train_steps_per_second': 4.91, 'total_flos': 1.2176307693699072e+16, 'train_loss': 0.14267427842881944, 'epoch': 3.0})

In [12]:
# Оценка на тестовой выборке
test_results = trainer.evaluate(tokenized_dataset["test"])
print("Accuracy на тестовой выборке:", test_results["eval_accuracy"])

# Сохранение модели
trainer.save_model("./ag_news_model")


Epoch,Training Loss,Validation Loss,Accuracy
0,No log,0.215107,0.945132


Accuracy на тестовой выборке: 0.9451315789473684


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
from transformers import pipeline

# Загружаем обученную модель
classifier = pipeline(
    "text-classification",
    model="./ag_news_model"
)

# Примеры новостей
news_samples = [
    "The stock market crashed after unexpected economic data.",
    "The local team won the championship after a thrilling final.",
    "Scientists discovered a new particle in a physics experiment.",
    "The president met with foreign leaders to discuss trade."
]

# Предсказания
for news in news_samples:
    prediction = classifier(news)[0]

    print("\nТекст:", news)
    print("Предсказанный класс:", prediction["label"])
    print("Уверенность модели:", round(prediction["score"], 4))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Текст: The stock market crashed after unexpected economic data.
Предсказанный класс: LABEL_1
Уверенность модели: 0.5788

Текст: The local team won the championship after a thrilling final.
Предсказанный класс: LABEL_1
Уверенность модели: 0.6282

Текст: Scientists discovered a new particle in a physics experiment.
Предсказанный класс: LABEL_1
Уверенность модели: 0.6093

Текст: The president met with foreign leaders to discuss trade.
Предсказанный класс: LABEL_1
Уверенность модели: 0.6093
